<a href="https://colab.research.google.com/github/Hershey0626/Website-Fingerprinting-Attack/blob/main/Website_Fingerprinting_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Website Fingerprinting: Phase 1 - Loading and Preprocessing Data

In this notebook, we load, inspect, and clean the traffic traces from the open-source **WFlib Benchmark Dataset Closed-World** archive (CW.npz) hosted on Zenodo. The objective is to standardize raw network packet capture characteristics into a uniform sequential format optimized for machine learning models.

1. 105730 total traffic traces (N)
2. 95 unique number of websites (C)

To prepare inputs for both our baseline Support Vector Machine (SVM) and primary 1D-Convolutional Neural Network (1D-CNN), we apply the following constraints to the feature matrix:

1. Direction Extraction: Convert all floating-point entries into binary direction indicators by mapping outgoing packets to +1.0, and incoming packets to -1.0.

2. Fixed-length Standardiziation: Standardize every packet sequence to a fixed length of $5,000$ elements. Traces exceeding this threshold are truncated, while shorter traces are zero-padded.

## Step 1: Loading Data from Closed-World Archive

Before beginning the project, here is the link to access the closed-world archive from Zenodo:
https://zenodo.org/records/13731720

Once downloaded on your local computer, drag the file into the Colab Files tab to upload the dataset. Then, unzip the archive and extract the feature matrix X and target labels y.

Finally, print the total traffic traces, label shapes, and the number of unique target websites, and inspect a single raw sample from the trace portfolio to verify the structure.

In [ ]:
!unzip data/CW.npz.zip -d data/

import numpy as np

# Load the archive using the exact path revealed by your unzip command
data_path = 'data/datasets/Defense/CW.npz'
data_archive = np.load(data_path, allow_pickle=True)

# Extract the feature matrix X and target labels y
X_raw = data_archive['X']
y_raw = data_archive['y']

# Print out the dataset statistics for your progress report
print("\n\nDataset Statistics: ")
print(f"Total Traffic Traces (Rows): {X_raw.shape[0]}")
print(f"Target Labels Shape: {y_raw.shape[0]}")
print(f"Number of unique websites (classes): {len(np.unique(y_raw))}")

# Inspect a single raw sample trace profile
print("\n\nRaw Sample Inspection: ")
print(f"Shape of the very first raw trace: {X_raw[0].shape}")
print(f"First 10 elements of sample 0: {X_raw[0][:10]}")

Archive:  data/CW.npz.zip
replace data/datasets/Defense/CW.npz? [y]es, [n]o, [A]ll, [N]one, [r]ename: Dataset Statistics: 
Total Traffic Traces (Rows): 105730
Target Labels Shape: 105730
Number of unique websites (classes): 95


Raw Sample Inspection: 
Shape of the very first raw trace: (10000,)
First 10 elements of sample 0: [ 1.00000000e-06 -9.28558873e-01  9.99152945e-01 -1.03146009e+00
  3.76949601e+00 -3.89432101e+00  3.89727907e+00 -4.02370195e+00
  4.03020387e+00 -4.15452604e+00]


## Step 2: Preprocessing Data from Closed-World Archive

In this section, we preprocess the raw traffic traces to isolate packet direction and standardize input dimensions for our machine learning models.

Traditional website fingerprinting models focus primarily on the sequence of packet directions rather than exact timestamps or variable packet sizes. To achieve this, we apply two main transformations:
1. **Direction Extraction:** We use a sign function to convert all raw floating-point magnitudes into binary indicators—mapping outbound packets to `+1.0` and inbound packets to `-1.0`.
2. **Fixed-Length Standardization:** To accommodate neural network input structures, we standardize every sequence to a fixed length of `5,000` elements. Traces longer than 5,000 are truncated, while shorter traces are zero-padded (`0.0`)

We use an optimized loop to build a uniform feature matrix of shape `(105730, 5000)` using `float32` to efficiently manage Google Colab's RAM.

In [ ]:
import numpy as np

print("Standardizing sequences to length 5000 and extracting direction vectors...")

# Define the target length
MAX_LEN = 5000

# Initialize clean matrix with float32 to conserve Colab RAM memory
X_clean = np.zeros((X_raw.shape[0], MAX_LEN), dtype=np.float32)

# Loop through and standardize each traffic trace profile
for i in range(X_raw.shape[0]):
    directions = np.sign(X_raw[i])

    length_to_copy = min(len(directions), MAX_LEN)

    # Insert the trace into our clean matrix (any remaining space stays 0 padding)
    X_clean[i, :length_to_copy] = directions[:length_to_copy]

print("\nPreprocessing Complete: ")
print(f"Final Cleaned Feature Matrix Shape: {X_clean.shape}")
print(f"First 10 elements of cleaned sample 0: {X_clean[0][:10]}")

Standardizing sequences to length 5000 and extracting direction vectors...

--- Preprocessing Complete ---
Final Cleaned Feature Matrix Shape: (105730, 5000)
First 10 elements of cleaned sample 0: [ 1. -1.  1. -1.  1. -1.  1. -1.  1. -1.]
